In [ ]:
import keras
import tensorflow as tf
import numpy as np

In [ ]:
# 1. ANALISI DEL PROTOTIPO (Shakespeare): 

# ACQUISIZIONE E PRE-PROCESSING DEI DATI
# Scarico il dataset "Tiny Shakespeare" direttamente dai server Google
path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt', 
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

# Lettura del testo e decodifica in formato stringa UTF-8
text_shakespeare = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f'Lunghezza del testo: {len(text_shakespeare)} caratteri')
print("\nEsempio (primi 250 caratteri):")
print(text_shakespeare[:250])  

def elabora_dizionari(text):
    """
    Costruisce il vocabolario di caratteri unici presenti nel testo e i
    dizionari di conversione carattere <-> indice numerico.
    """
    vocab = sorted(set(text))   # lista dei caratteri unici (ordinati)

    char2idx = {u:i for i, u in enumerate(vocab)}  # dizionario carattere→indice numerico
    idx2char = np.array(vocab)    # array inverso indice→carattere

    text_as_int = np.array([char2idx[c] for c in text])    # testo codificato come sequenza di interi

    return vocab, char2idx, idx2char, text_as_int

vocab_shakespeare, char2idx_shakespeare, idx2char_shakespeare, text_as_int_shakespeare = elabora_dizionari(text_shakespeare)
n_char_unici_shakespeare = len(vocab_shakespeare)

print(f"\n>>> Numero di caratteri unici usati da Shakespeare: {n_char_unici_shakespeare}")
print("Vocabolario:", vocab_shakespeare)

Lunghezza del testo: 1115394 caratteri

Esempio (primi 250 caratteri):
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.


>>> Numero di caratteri unici usati da Shakespeare: 65
Vocabolario: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [5]:
# CREAZIONE DEL DATASET CON TF.DATA
seq_length = 250    # lunghezza di ogni sequenza di training
BATCH_SIZE = 64
BUFFER_SIZE = 10000   # per lo shuffle del dataset
embedding_dim = 256
rnn_units = 1024
EPOCHS = 30       

def crea_dataset_addestramento(text_as_int, seq_length, vocab_size,
                                batch_size=BATCH_SIZE, buffer_size=BUFFER_SIZE):
    """
    A partire dal testo codificato come interi, costruisce il dataset
    di addestramento fatto di coppie (input, target) shiftate di un carattere,
    raggruppate in sequenze di lunghezza seq_length e in batch.
    """
    examples_per_epoch = len(text_as_int) // (seq_length + 1)
    print(f"Esempi per epoca: {examples_per_epoch}")

    char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)      # Trasforma l'array piatto di interi in un dataset TF
    # 
    sequences = char_dataset.batch(seq_length + 1, drop_remainder=True) # drop_remainder=True scarta l'ultimo blocco se più corto, per avere batch di forma uniforme

    def split_input_target(chunk):
        """
        Divide il chunk in input (caratteri da 0 a L-1) e target (da 1 a L)
        Esempio: Input 'Hell' -> Target 'ello'
        """
        input_text = chunk[:-1]
        target_text = chunk[1:]
        return input_text, target_text

    dataset = sequences.map(split_input_target)
    dataset = dataset.shuffle(buffer_size).batch(batch_size, drop_remainder=True)   # shuffle() mescola le sequenze per evitare che il modello impari l'ordine fisso del testo + batch() raggruppa le sequenze in batch da 64

    return dataset


vocab_size_shakespeare = len(vocab_shakespeare)
dataset_shakespeare = crea_dataset_addestramento(
    text_as_int_shakespeare, seq_length, vocab_size_shakespeare
)
print(dataset_shakespeare)

Esempi per epoca: 4443
<_BatchDataset element_spec=(TensorSpec(shape=(64, 250), dtype=tf.int64, name=None), TensorSpec(shape=(64, 250), dtype=tf.int64, name=None))>


In [ ]:
# DEFINIZIONE DEL MODELLO E TRAINING

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    """Costruisce il modello char-RNN: Embedding -> GRU -> Dense."""
    model = keras.Sequential([
        keras.Input(shape=(None,)),   # sequenze di lunghezza variabile
        keras.layers.Embedding(vocab_size, embedding_dim),
        keras.layers.GRU(rnn_units, return_sequences=True, stateful=False, recurrent_initializer='glorot_uniform'),  # 'stateful=True' permette di ricordare il contesto tra batch successivi durante l'inferenza
        keras.layers.GRU(rnn_units, return_sequences=True, stateful=False),  # 2° layer GRU per catturare concetti più complessi
        keras.layers.Dense(vocab_size)  # Dense: produce i 'logits'(punteggi grezzi per ogni carattere nel vocabolario)
    ])
    return model

def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)    # SparseCategoricalCrossentropy è la loss corretta quando i target sono interi (indici di classe) + from_logits=True dice alla loss di applicare internamente la softmax


model_shakespeare = build_model(
    vocab_size=vocab_size_shakespeare,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
    batch_size=BATCH_SIZE
)
model_shakespeare.summary()
model_shakespeare.compile(optimizer='adam', loss=loss)

history_shakespeare = model_shakespeare.fit(
    dataset_shakespeare, epochs=EPOCHS)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, None, 256)      │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, None, 1024)     │     3,938,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, None, 1024)     │     6,297,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, None, 65)       │        66,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,319,169 (39.36 MB)

 Trainable params: 10,319,169 (39.36 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# FUNZIONE DI GENERAZIONE CON TEMPERATURA

def generate_text(model, start_string, char2idx, idx2char, temperature=1.0, num_generate=400):
    """
    Genera testo carattere per carattere a partire da start_string.

    temperature:
        - bassa (es. 0.2)  -> output piu' "conservativo"/prevedibile
        - alta  (es. 1.0-2.0) -> output piu' "creativo"/rischioso
    """
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)  # (1, lunghezza_iniziale)

    text_generated = []

    for _ in range(num_generate):
        predictions = model(input_eval)       # (1, lunghezza, vocab_size)
        predictions = tf.squeeze(predictions, 0)      # (lunghezza, vocab_size)
        predictions = predictions[-1:] / temperature  # solo l'ultimo step e divide i logits per la temperatura. 
            
        predicted_id = tf.random.categorical(predictions, num_samples=1)[0, 0].numpy() # Campiona un carattere dalla distribuzione di probabilità

        # SLIDING WINDOW: aggiungo il carattere predetto alla sequenza e la ripasso al modello
        input_eval = tf.concat([input_eval, [[predicted_id]]], axis=1)
        text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))


def main(model, start_string, char2idx, idx2char, temperature=1.0, num_generate=400):
    return generate_text(model, start_string, char2idx, idx2char,
                         temperature=temperature, num_generate=num_generate)


# Esempio di utilizzo (con modello addestrato)
testo_generato_shakespeare = main(model_shakespeare, "ROMEO: ", char2idx_shakespeare, idx2char_shakespeare, temperature=1.0)
print(testo_generato_shakespeare)

In [9]:
# 2. CAMBIO DEL DATASET (Dante Alighieri) --> suggerimento codice come da traccia
import requests
import urllib3

# Disabilitiamo i messaggi di avviso "InsecureRequestWarning" per avere un output pulito
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


def scarica_testo_dante():
    """
    Scarica la Divina Commedia ignorando gli errori di certificato SSL.
    """
    url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
    print(f"Scaricamento testo da: {url}...")

    try:
        # AGGIUNTA FONDAMENTALE: verify=False
        # Dice a Python: "Fidati del server anche se non riconosci il certificato"
        response = requests.get(url, verify=False)
        response.raise_for_status()

        response.encoding = 'utf-8'

        text = response.text
        print(f"Download completato. Lunghezza testo: {len(text)} caratteri")
        print(f"Esempio (primi 100 caratteri):\n{text[:100]}")
        return text

    except Exception as e:
        print(f"Errore ancora presente: {e}")

        # Se fallisce anche cosi', usiamo un link alternativo di GitHub (spesso piu' compatibile)
        print("Tentativo con link di riserva (GitHub)...")
        try:
            url_backup = "https://raw.githubusercontent.com/wpm/t-snetext-vis/master/data/divina_commedia.txt"
            r2 = requests.get(url_backup, verify=False)
            r2.encoding = 'utf-8'
            return r2.text
        except Exception:
            return None


# Eseguiamo il download
text_dante = scarica_testo_dante()

vocab_dante, char2idx_dante, idx2char_dante, text_as_int_dante = elabora_dizionari(text_dante)
n_char_unici_dante = len(vocab_dante)

print("\nCONFRONTO")
print(f">>> Numero di caratteri unici usati da Shakespeare: {n_char_unici_shakespeare}")
print(f"\n>>> Numero di caratteri unici usati da Dante: {n_char_unici_dante}\n")
print("Vocabolario Dante:", vocab_dante)

Scaricamento testo da: https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt...
Download completato. Lunghezza testo: 551846 caratteri
Esempio (primi 100 caratteri):
LA DIVINA COMMEDIA
di Dante Alighieri
INFERNO



Inferno: Canto I

  Nel mezzo del cammin di

CONFRONTO
>>> Numero di caratteri unici usati da Shakespeare: 65

>>> Numero di caratteri unici usati da Dante: 69

Vocabolario Dante: ['\n', '\r', ' ', '!', '"', "'", '(', ')', ',', '-', '.', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'X', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'x', 'y', 'z', '~', 'à', 'è', 'é', 'ì', 'ï', 'ò', 'ó', 'ù']


N.B.

Caratteri unici: 65 (Shakespeare) vs 69 (Dante)

La differenza di 4 caratteri è coerente con l'aspettativa della traccia: sono quasi certamente gli accenti italiani (à, è, ì, ò, ù)


In [10]:
# ADDESTRAMENTO PARAMETRI 
seq_length_dante = 80   # riduzione lunghezza sequenza perchè Dante scrive in terzine, più brevi

vocab_size_dante = len(vocab_dante)
dataset_dante = crea_dataset_addestramento(
    text_as_int_dante, seq_length_dante, vocab_size_dante
)
print(dataset_dante)

Esempi per epoca: 6812
<_BatchDataset element_spec=(TensorSpec(shape=(64, 80), dtype=tf.int64, name=None), TensorSpec(shape=(64, 80), dtype=tf.int64, name=None))>


In [ ]:
# DEFINIZIONE DEL MODELLO E TRAINING

model_dante = build_model(
    vocab_size=vocab_size_dante,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
    batch_size=BATCH_SIZE
)
model_dante.summary()
model_dante.compile(optimizer='adam', loss=loss)

history_dante = model_dante.fit(
    dataset_dante, epochs=EPOCHS)

N.B.

Riducendo seq_length da 250 (usato per Shakespeare) a 80 (Dante) il tempo per singolo step di training è sceso da 391ms a 125ms circa.

Una riduzione di circa 3 volte che corrisponde quasi esattamente al rapporto tra le due lunghezze di sequenza (250/80 ≈ 3,13). 

Questo conferma che il costo computazionale per step cresce linearmente con seq_length, perché la GRU deve eseguire un numero di passaggi di backpropagation-through-time proporzionale alla lunghezza della sequenza processata.


In [ ]:
# FUNZIONE DI GENERAZIONE E TUNING DELLA TEMPERATURA
prompt_dante = "Nel mezzo "

for temp in [0.2, 1.0, 2.0]:
    print(f"\n{'='*20} TEMPERATURA = {temp} {'='*20}\n")
    testo_generato_dante = main(
        model=model_dante,
        start_string=prompt_dante,
        char2idx=char2idx_dante,
        idx2char=idx2char_dante,
        temperature=temp,
        num_generate=400
    )
    print(testo_generato_dante)

N.B.

==================== TEMPERATURA = 0.2 ====================

Nel mezzo per forza mi fioce,

tal che di qua dal rio mi fé paura.

  Poi vidi quattro in umile paruta;

e sì volea con esse e odi e credi;

ché la via di fuor del monte avanza.

  Vedea colui che fu nobil creato

più male avesser uon che tu lai

vidi gente che di fece trestina

la vista che riceve il vostro andava.

  "Oh!", diss'io lui, "non se' tu Oderisi,

come del sole stella mattutina stella

che la m

==================== TEMPERATURA = 1.0 ====================

Nel mezzo e un qua giù la pone".

  Alto sospir, che duolo strisse lungo;

per ch'io divenni tal, quando lo 'ntelletto.

  Quinci si mosse spirito a nomarlo

de la mia donna, e l'animo con essi,

e da ogne antica e con inferne bianche,

sola per cammin colpa dov'è più bella,

  orando grazia oltre nostre scarne,

tal ne la fiamma del sol per trasparere

lo lume suo per piacer di piglio.

  Oh abbia ben ch'a

==================== TEMPERATURA = 2.0 ====================

Nel mezzo plivié

e così a sé l'ance? Tal eraggi 'l cielo;

e tanta augia la sposa soverchia;

  esso Beatrice, vesta circudendo;

quel maremente offe Sanza ne diso;

nai, se come abino mi diè,

se' nullo spirto di tal trafitta?

 perché lai non si è da sua lucenti,

e quanto il sol Ber sì fuor di chiusura più ampa,

che per proferta poi che lagrimare l'ira,

per che 'l morinza rigià e li apvivazi.

  O be






1. Temp = 0.2 (freddo/conservativo)

    Non ci sono loop ripetitivi (nessuna parola/frase incollata più volte).
    
    Il testo è il più "pulito": rime quasi plausibili, struttura di endecasillabo abbastanza rispettata, punteggiatura e a capo coerenti con lo stile dantesco. 
    
    Grammaticalmente è il più solido dei tre


2. Temp = 1.0 (standard)

    Sintassi e grammatica reggono ancora abbastanza bene — frasi di senso compiuto, costruzioni tipiche dantesche, ma iniziano a comparire licenze più libere e qualche costruzione più ardita rispetto a 0.2. 
    
    È il compromesso tra fedeltà grammaticale e varietà lessicale


3. Temp = 2.0 (caldo/folle)

    Qui iniziano a comparire parole inventate/non italiane ("plivié", "eraggi", "augia", "circudendo", "Ber", "morinza", "apvivazi") — esattamente la "lingua aliena" descritta nella traccia. 
    
    La sintassi si spezza più spesso, la punteggiatura diventa erratica.